In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('data/dataprocessing_v2.csv')
df.head()

,Brand,Series,Version,Max Tension,Origin,Price,Tech_Frame,Tech_Material,Tech_Stability
0,Lining,1210F,v1,28.0,China,1140000,OPTIMUM FRAME,"TB NANO, AEROTEC BEAM",AIR SYSTEM
1,Lining,3D,v1,30.0,China,1150000,OPTIMUM FRAME,"TB NANO, AEROTEC BEAM","WING STABILIZER, 3D CALIBAR, STABILIZED TORSIO..."
2,Lining,3D,v1,30.0,China,1150000,OPTIMUM FRAME,"TB NANO, AEROTEC BEAM, CARBON","WING STABILIZER, 3D CALIBAR"
3,Lining,3D,v1,28.0,China,1205000,NaN,NaN,NaN
4,Lining,3D,v1,30.0,China,1450000,OPTIMUM FRAME,"TB NANO, CARBON","WING STABILIZER, 3D CALIBAR, STABILIZED TORSIO..."


In [3]:
df = df.drop(columns=['Max Tension'])
print(df.columns.tolist())
print(df.shape)

df.to_csv('data/dataprocessing_v2.csv', index=False)

['Brand', 'Series', 'Version', 'Origin', 'Price', 'Tech_Frame', 'Tech_Material', 'Tech_Stability']
(1067, 8)


In [4]:
from sklearn.preprocessing import MultiLabelBinarizer

def encode_tech_column(df, col_name):
    
    # Điền NaN bằng chuỗi rỗng, tách bằng dấu phẩy
    split_data = df[col_name].fillna('').apply(
        lambda x: [tech.strip() for tech in x.split(',') if tech.strip() != '']
    )
    
    mlb = MultiLabelBinarizer()
    encoded = mlb.fit_transform(split_data)
    
    # Tạo DataFrame với tên cột = tên công nghệ
    encoded_df = pd.DataFrame(
        encoded,
        columns=[f"{col_name}_{cls}" for cls in mlb.classes_],
        index=df.index
    )
    return encoded_df

# Áp dụng cho từng cột Tech
tech_cols = ['Tech_Frame', 'Tech_Material', 'Tech_Stability']

encoded_parts = []
for col in tech_cols:
    encoded_parts.append(encode_tech_column(df, col))

# Ghép vào dataframe, xóa cột cũ
df_encoded = pd.concat([df.drop(columns=tech_cols)] + encoded_parts, axis=1)

df_encoded.to_csv('data/dataprocessing_encoded.csv', index=False)


print(f"Shape: {df_encoded.shape}")
print(f"Columns ({len(df_encoded.columns)}):", df_encoded.columns.tolist())
df_encoded.head()


Shape: (1067, 111)
Columns (111): ['Brand', 'Series', 'Version', 'Origin', 'Price', 'Tech_Frame_AERO FRAME', 'Tech_Frame_AERO-BOX FRAME', 'Tech_Frame_AERO-DIAMOND', 'Tech_Frame_AERO-SWORD', 'Tech_Frame_BOX WING FRAME', 'Tech_Frame_CATAPULT STRUCTURE', 'Tech_Frame_COMPACT FRAME', 'Tech_Frame_DOURA GROMMENT', 'Tech_Frame_EXPANDED SWEET SPOT FOR CONTROL', 'Tech_Frame_FULL CONE', 'Tech_Frame_ISOMETRIC', 'Tech_Frame_New GROMMET PATTERN', 'Tech_Frame_OCTABALDE', 'Tech_Frame_OPTIMUM FRAME', 'Tech_Frame_OVAL HEAD', 'Tech_Frame_POCKETING BOOSTER', 'Tech_Frame_POWER BOX', 'Tech_Frame_SHARP WIND', 'Tech_Frame_SONIC BOOM SYSTEM', 'Tech_Frame_SWORD', 'Tech_Frame_TECTONIC TECHNOLOGY', 'Tech_Frame_TRI-FORMATION', 'Tech_Material_ACC-RIF TECH', 'Tech_Material_AEROTEC BEAM', 'Tech_Material_CARBON', 'Tech_Material_FIBER REINFORCED SYSTEM', 'Tech_Material_FRS', 'Tech_Material_FRTP TECH', 'Tech_Material_HARD CORED TECHNOLOGY', 'Tech_Material_HDF', 'Tech_Material_HOT MELT', 'Tech_Material_HYBRID CN', 'Tech_

,Brand,Series,Version,Origin,Price,Tech_Frame_AERO FRAME,Tech_Frame_AERO-BOX FRAME,Tech_Frame_AERO-DIAMOND,Tech_Frame_AERO-SWORD,Tech_Frame_BOX WING FRAME,...,Tech_Stability_TFA CAP PLUS,Tech_Stability_THUNDER TECHNOLOGY,Tech_Stability_TRANS-WEIGHT SYSTEM,Tech_Stability_TRI-IBUMPER,Tech_Stability_TRI-VOLTAGE SYSTEM,Tech_Stability_TURBO CHARGING,Tech_Stability_VIBRATION DAMPENING MESH,Tech_Stability_WES,Tech_Stability_WHIPPING,Tech_Stability_WING STABILIZER
0,Lining,1210F,v1,China,1140000,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,Lining,3D,v1,China,1150000,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
2,Lining,3D,v1,China,1150000,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
3,Lining,3D,v1,China,1205000,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,Lining,3D,v1,China,1450000,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1


In [5]:
# One-Hot Encoding cho Brand
df_encoded = pd.get_dummies(df_encoded, columns=['Brand'], dtype=int)

print(df_encoded[['Brand_Lining', 'Brand_Victor', 'Brand_Yonex']].head())
print(df_encoded.shape)


   Brand_Lining  Brand_Victor  Brand_Yonex
0             1             0            0
1             1             0            0
2             1             0            0
3             1             0            0
4             1             0            0
(1067, 113)


In [6]:
version_map = {'v1': 1, 'v2': 2, 'v3': 3}
df_encoded['Version'] = df_encoded['Version'].map(version_map)
print(df_encoded['Version'].value_counts())

Version
1    432
2    395
3    240
Name: count, dtype: int64


In [7]:
# One-Hot cho Origin
if 'Origin' in df_encoded.columns:
    origin_dummies = pd.get_dummies(df_encoded['Origin'], prefix='Origin', dtype=int)
    df_encoded = pd.concat([df_encoded, origin_dummies], axis=1)
    df_encoded = df_encoded.drop(columns=['Origin'])
else:
    print("Cột 'Origin' đã bị encode rồi, bỏ qua.")
df_encoded.head()



,Series,Version,Price,Tech_Frame_AERO FRAME,Tech_Frame_AERO-BOX FRAME,Tech_Frame_AERO-DIAMOND,Tech_Frame_AERO-SWORD,Tech_Frame_BOX WING FRAME,Tech_Frame_CATAPULT STRUCTURE,Tech_Frame_COMPACT FRAME,...,Tech_Stability_VIBRATION DAMPENING MESH,Tech_Stability_WES,Tech_Stability_WHIPPING,Tech_Stability_WING STABILIZER,Brand_Lining,Brand_Victor,Brand_Yonex,Origin_China,Origin_Japan,Origin_Taiwan
0,1210F,1,1140000,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,1,0,0
1,3D,1,1150000,0,0,0,0,0,0,0,...,0,0,0,1,1,0,0,1,0,0
2,3D,1,1150000,0,0,0,0,0,0,0,...,0,0,0,1,1,0,0,1,0,0
3,3D,1,1205000,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,1,0,0
4,3D,1,1450000,0,0,0,0,0,0,0,...,0,0,0,1,1,0,0,1,0,0


In [8]:
# Target Encoding cho Series
series_mean = df_encoded.groupby('Series')['Price'].mean()  
df_encoded['Series_encoded'] = df_encoded['Series'].map(series_mean)
df_encoded = df_encoded.drop(columns=['Series'])

min_val = df_encoded['Series_encoded'].min()    # Normalize
max_val = df_encoded['Series_encoded'].max()
df_encoded['Series_encoded'] = (df_encoded['Series_encoded'] - min_val) / (max_val - min_val)

df_encoded.head()


,Version,Price,Tech_Frame_AERO FRAME,Tech_Frame_AERO-BOX FRAME,Tech_Frame_AERO-DIAMOND,Tech_Frame_AERO-SWORD,Tech_Frame_BOX WING FRAME,Tech_Frame_CATAPULT STRUCTURE,Tech_Frame_COMPACT FRAME,Tech_Frame_DOURA GROMMENT,...,Tech_Stability_WES,Tech_Stability_WHIPPING,Tech_Stability_WING STABILIZER,Brand_Lining,Brand_Victor,Brand_Yonex,Origin_China,Origin_Japan,Origin_Taiwan,Series_encoded
0,1,1140000,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,1,0,0,0.054200
1,1,1150000,0,0,0,0,0,0,0,0,...,0,0,1,1,0,0,1,0,0,0.134144
2,1,1150000,0,0,0,0,0,0,0,0,...,0,0,1,1,0,0,1,0,0,0.134144
3,1,1205000,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,1,0,0,0.134144
4,1,1450000,0,0,0,0,0,0,0,0,...,0,0,1,1,0,0,1,0,0,0.134144


In [ ]:
# Xóa các cột tech có variance thấp

import numpy as np

tech_cols = [c for c in df_encoded.columns if c.startswith('Tech_')]

# Tính variance của từng cột
variances = {c: df_encoded[c].var() for c in tech_cols}

threshold = 0.01  # cột nào var < 1% thì bỏ
low_var_cols = [c for c, v in variances.items() if v < threshold]
df_encoded = df_encoded.drop(columns=low_var_cols)

print(f"Bỏ {len(low_var_cols)} cột variance thấp")

df_encoded.head()

Bỏ 33 cột variance thấp


,Version,Price,Tech_Frame_AERO FRAME,Tech_Frame_AERO-BOX FRAME,Tech_Frame_AERO-DIAMOND,Tech_Frame_AERO-SWORD,Tech_Frame_BOX WING FRAME,Tech_Frame_COMPACT FRAME,Tech_Frame_ISOMETRIC,Tech_Frame_New GROMMET PATTERN,...,Tech_Stability_WES,Tech_Stability_WHIPPING,Tech_Stability_WING STABILIZER,Brand_Lining,Brand_Victor,Brand_Yonex,Origin_China,Origin_Japan,Origin_Taiwan,Series_encoded
0,1,1140000,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,1,0,0,0.054200
1,1,1150000,0,0,0,0,0,0,0,0,...,0,0,1,1,0,0,1,0,0,0.134144
2,1,1150000,0,0,0,0,0,0,0,0,...,0,0,1,1,0,0,1,0,0,0.134144
3,1,1205000,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,1,0,0,0.134144
4,1,1450000,0,0,0,0,0,0,0,0,...,0,0,1,1,0,0,1,0,0,0.134144


In [10]:
# Sắp xếp lại các cột
series_cols  = ['Series_encoded'] if 'Series_encoded' in df_encoded.columns else []
brand_cols   = [c for c in df_encoded.columns if c.startswith('Brand_')]
origin_cols  = [c for c in df_encoded.columns if c.startswith('Origin_')]
version_col  = ['Version']
price_col    = ['Price']
tech_cols    = [c for c in df_encoded.columns if c.startswith('Tech_')]

new_order = series_cols + version_col + brand_cols + origin_cols + price_col + tech_cols
df_encoded = df_encoded[new_order]
print(df_encoded.columns.tolist()[:10])
df_encoded.head()

['Series_encoded', 'Version', 'Brand_Lining', 'Brand_Victor', 'Brand_Yonex', 'Origin_China', 'Origin_Japan', 'Origin_Taiwan', 'Price', 'Tech_Frame_AERO FRAME']


,Series_encoded,Version,Brand_Lining,Brand_Victor,Brand_Yonex,Origin_China,Origin_Japan,Origin_Taiwan,Price,Tech_Frame_AERO FRAME,...,Tech_Stability_STABILIZED TORSION ANGLE,Tech_Stability_SUPER SLIM SHAFT,Tech_Stability_T-ANCHOR,Tech_Stability_TERS,Tech_Stability_TRI-IBUMPER,Tech_Stability_TRI-VOLTAGE SYSTEM,Tech_Stability_TURBO CHARGING,Tech_Stability_WES,Tech_Stability_WHIPPING,Tech_Stability_WING STABILIZER
0,0.054200,1,1,0,0,1,0,0,1140000,0,...,0,0,0,0,0,0,0,0,0,0
1,0.134144,1,1,0,0,1,0,0,1150000,0,...,1,0,0,0,0,0,0,0,0,1
2,0.134144,1,1,0,0,1,0,0,1150000,0,...,0,0,0,0,0,0,0,0,0,1
3,0.134144,1,1,0,0,1,0,0,1205000,0,...,0,0,0,0,0,0,0,0,0,0
4,0.134144,1,1,0,0,1,0,0,1450000,0,...,1,0,0,0,0,0,0,0,0,1


In [11]:
df_encoded.to_csv('data/dataprocessing_encoded.csv', index=False)
print(f"Đã lưu: {df_encoded.shape[0]} rows × {df_encoded.shape[1]} cols")
print(df_encoded.columns.tolist())


Đã lưu: 1067 rows × 82 cols
['Series_encoded', 'Version', 'Brand_Lining', 'Brand_Victor', 'Brand_Yonex', 'Origin_China', 'Origin_Japan', 'Origin_Taiwan', 'Price', 'Tech_Frame_AERO FRAME', 'Tech_Frame_AERO-BOX FRAME', 'Tech_Frame_AERO-DIAMOND', 'Tech_Frame_AERO-SWORD', 'Tech_Frame_BOX WING FRAME', 'Tech_Frame_COMPACT FRAME', 'Tech_Frame_ISOMETRIC', 'Tech_Frame_New GROMMET PATTERN', 'Tech_Frame_OPTIMUM FRAME', 'Tech_Frame_POWER BOX', 'Tech_Frame_SWORD', 'Tech_Frame_TRI-FORMATION', 'Tech_Material_AEROTEC BEAM', 'Tech_Material_CARBON', 'Tech_Material_FIBER REINFORCED SYSTEM', 'Tech_Material_FRS', 'Tech_Material_FRTP TECH', 'Tech_Material_HARD CORED TECHNOLOGY', 'Tech_Material_HDF', 'Tech_Material_HOT MELT', 'Tech_Material_LOCKING CUBIC', 'Tech_Material_M40X', 'Tech_Material_METALLIC', 'Tech_Material_MPCF REINFORCING TECHNOLOGY', 'Tech_Material_NAMD', 'Tech_Material_NANO FORTIFY', 'Tech_Material_NANO TEC', 'Tech_Material_NANOCELL NEO', 'Tech_Material_NANOMESH NEO', 'Tech_Material_NANOSCIENC